# 一、合并基因表达和DFS

In [1]:
import numpy as np
import pandas as pd

# Load EDASeq normalized data.
allPatients = pd.read_csv('../data/EDAseqNormalization.csv')
allPatients

,Unnamed: 0,TCGA-CV-A45W-01A-11R-A24Z-07,TCGA-CR-7388-01A-11R-2016-07,TCGA-CR-7379-01A-11R-2016-07,TCGA-CR-7393-01A-11R-2016-07,TCGA-P3-A6T2-01A-11R-A34R-07,TCGA-CN-4737-01A-01R-1436-07,TCGA-CV-6960-01A-41R-2016-07,TCGA-T3-A92N-01A-11R-A39I-07,TCGA-CV-A6JZ-01A-11R-A31N-07,...,TCGA-CV-6933-11A-01R-1915-07,TCGA-CV-7242-11A-01R-2016-07,TCGA-CV-7235-11A-01R-2016-07,TCGA-CV-7252-11A-01R-2016-07,TCGA-CV-7097-11A-01R-2016-07,TCGA-CV-7438-11A-01R-2132-07,TCGA-CV-7406-11A-01R-2081-07,TCGA-CV-6943-11A-01R-1915-07,TCGA-CV-7434-11A-01R-2132-07,TCGA-CV-7437-11A-01R-2132-07
0,A1BG,5,10,4,7,11,3,27,5,17,...,3,8,4,2,7,0,1,0,3,1
1,A1CF,0,1,0,0,4,0,0,0,0,...,2,0,1,0,0,0,0,0,0,0
2,A2ML1,28576,9911,20409,121449,4051,27739,12192,1764,1197,...,154,77300,25,178991,67,95179,31805,130272,36,103654
3,A2M,16551,21816,17747,34953,4491,21035,2671,3468,14889,...,29269,5541,87967,7749,76388,5868,22216,4598,194354,2637
4,A4GALT,27648,2143,7844,2996,9398,3412,4115,1568,507,...,2324,7207,4485,4220,2836,2808,2723,2789,4043,6137
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17495,ZXDC,2210,2470,1332,1384,1482,3035,1726,2433,4249,...,1372,1588,1382,1666,1176,1572,1458,1440,1436,1789
17496,ZYG11A,193,31,27,27,6,69,530,19,10,...,4,83,7,76,4,92,19,115,7,76
17497,ZYG11B,1252,2236,1448,2241,1349,1761,2241,1108,1276,...,2723,1820,2135,2572,4427,1788,7850,1990,2488,1694
17498,ZYX,10052,14475,14846,20793,19369,12233,7640,22495,17386,...,6641,15119,27471,11721,16665,16694,17913,11556,18586,13443


In [2]:
# Make sure there are no duplicate genes
for i in allPatients.iloc[:, 0].duplicated():
    if i == True:
        print(False)

In [3]:
allPatients = allPatients.T
allPatients.columns = allPatients.iloc[0, :]
allPatients.insert(0, 'submitter', allPatients.index)
allPatients

Unnamed: 0,submitter,A1BG,A1CF,A2ML1,A2M,A4GALT,A4GNT,AAAS,AACS,AADACL2,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
Unnamed: 0,Unnamed: 0,A1BG,A1CF,A2ML1,A2M,A4GALT,A4GNT,AAAS,AACS,AADACL2,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
TCGA-CV-A45W-01A-11R-A24Z-07,TCGA-CV-A45W-01A-11R-A24Z-07,5,0,28576,16551,27648,0,2154,2439,161,...,1176,1554,1794,104,515,2210,193,1252,10052,2236
TCGA-CR-7388-01A-11R-2016-07,TCGA-CR-7388-01A-11R-2016-07,10,1,9911,21816,2143,7,1471,2716,7,...,781,992,1796,129,801,2470,31,2236,14475,3757
TCGA-CR-7379-01A-11R-2016-07,TCGA-CR-7379-01A-11R-2016-07,4,0,20409,17747,7844,0,2406,4061,2,...,1527,2404,2341,171,940,1332,27,1448,14846,3657
TCGA-CR-7393-01A-11R-2016-07,TCGA-CR-7393-01A-11R-2016-07,7,0,121449,34953,2996,0,1182,3677,25,...,1198,643,709,208,975,1384,27,2241,20793,3996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-CV-7438-11A-01R-2132-07,TCGA-CV-7438-11A-01R-2132-07,0,0,95179,5868,2808,0,1643,4355,220,...,1537,1314,1560,109,506,1572,92,1788,16694,3608
TCGA-CV-7406-11A-01R-2081-07,TCGA-CV-7406-11A-01R-2081-07,1,0,31805,22216,2723,0,1204,2576,9,...,1009,769,720,152,720,1458,19,7850,17913,3216
TCGA-CV-6943-11A-01R-1915-07,TCGA-CV-6943-11A-01R-1915-07,0,0,130272,4598,2789,0,1500,5622,1356,...,1514,1126,1529,106,643,1440,115,1990,11556,3610
TCGA-CV-7434-11A-01R-2132-07,TCGA-CV-7434-11A-01R-2132-07,3,0,36,194354,4043,0,1192,4491,2,...,648,438,400,164,906,1436,7,2488,18586,3415


In [4]:
allPatients.iloc[1:, 0] = allPatients.iloc[1:, 0].map(lambda x : '-'.join(x.split('-')[:4]))
allPatients

Unnamed: 0,submitter,A1BG,A1CF,A2ML1,A2M,A4GALT,A4GNT,AAAS,AACS,AADACL2,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
Unnamed: 0,Unnamed: 0,A1BG,A1CF,A2ML1,A2M,A4GALT,A4GNT,AAAS,AACS,AADACL2,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
TCGA-CV-A45W-01A-11R-A24Z-07,TCGA-CV-A45W-01A,5,0,28576,16551,27648,0,2154,2439,161,...,1176,1554,1794,104,515,2210,193,1252,10052,2236
TCGA-CR-7388-01A-11R-2016-07,TCGA-CR-7388-01A,10,1,9911,21816,2143,7,1471,2716,7,...,781,992,1796,129,801,2470,31,2236,14475,3757
TCGA-CR-7379-01A-11R-2016-07,TCGA-CR-7379-01A,4,0,20409,17747,7844,0,2406,4061,2,...,1527,2404,2341,171,940,1332,27,1448,14846,3657
TCGA-CR-7393-01A-11R-2016-07,TCGA-CR-7393-01A,7,0,121449,34953,2996,0,1182,3677,25,...,1198,643,709,208,975,1384,27,2241,20793,3996
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-CV-7438-11A-01R-2132-07,TCGA-CV-7438-11A,0,0,95179,5868,2808,0,1643,4355,220,...,1537,1314,1560,109,506,1572,92,1788,16694,3608
TCGA-CV-7406-11A-01R-2081-07,TCGA-CV-7406-11A,1,0,31805,22216,2723,0,1204,2576,9,...,1009,769,720,152,720,1458,19,7850,17913,3216
TCGA-CV-6943-11A-01R-1915-07,TCGA-CV-6943-11A,0,0,130272,4598,2789,0,1500,5622,1356,...,1514,1126,1529,106,643,1440,115,1990,11556,3610
TCGA-CV-7434-11A-01R-2132-07,TCGA-CV-7434-11A,3,0,36,194354,4043,0,1192,4491,2,...,648,438,400,164,906,1436,7,2488,18586,3415


In [5]:
# read DFS
all_dfs = pd.read_excel("../data/dfs.xlsx", usecols=[0, 4])
all_dfs = all_dfs.rename(columns={'submitter_id.samples': 'submitter'})
all_dfs

,submitter,DFS_MONTHS
0,TCGA-QK-A8Z9-01B,6.73
1,TCGA-CR-7370-01A,3.45
2,TCGA-CV-7250-01A,3.65
3,TCGA-IQ-A61L-01A,5.12
4,TCGA-CV-6936-01A,5.16
...,...,...
196,TCGA-CN-4741-01A,71.22
197,TCGA-CN-5360-01A,71.25
198,TCGA-P3-A6T2-01A,75.49
199,TCGA-CR-5243-01A,84.17


In [6]:
myDataset = pd.merge(allPatients, all_dfs, how='inner', on='submitter')
# The processed data is saved as a csv file, which can be transposed and saved easily for Excel to view
myDataset.T.to_csv('DataSheet.csv', sep='\t')
myDataset

,submitter,A1BG,A1CF,A2ML1,A2M,A4GALT,A4GNT,AAAS,AACS,AADACL2,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,DFS_MONTHS
0,TCGA-CR-7379-01A,4,0,20409,17747,7844,0,2406,4061,2,...,2404,2341,171,940,1332,27,1448,14846,3657,34.03
1,TCGA-CR-7393-01A,7,0,121449,34953,2996,0,1182,3677,25,...,643,709,208,975,1384,27,2241,20793,3996,32.62
2,TCGA-P3-A6T2-01A,11,4,4051,4491,9398,0,2046,1187,4,...,2569,5783,179,904,1482,6,1349,19369,2340,75.49
3,TCGA-CN-4737-01A,3,0,27739,21035,3412,4,1818,3932,98,...,1314,1162,120,543,3035,69,1761,12233,4602,20.53
4,TCGA-CV-5434-01A,5,1,2168,30339,3964,0,2050,1889,3,...,1430,1425,158,609,1419,5,1684,16838,3187,61.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,TCGA-BA-6869-01A,20,0,8139,4718,7458,0,1613,3445,1,...,1464,1631,191,604,2024,126,1628,9353,2023,21.16
181,TCGA-IQ-A61E-01A,12,0,232,5371,1683,8,1383,2690,0,...,2157,4726,93,322,1758,271,1161,26357,2837,37.68
182,TCGA-CN-6023-01A,9,3,324,13188,9383,0,2233,767,23,...,1671,3234,475,1185,1741,27,1603,8071,6268,52.04
183,TCGA-MT-A51X-01A,4,1,27382,7999,2810,0,844,2786,401,...,1395,1553,55,591,1971,48,1057,14412,2061,7.95


In [7]:
# Save Gene Name
gene = myDataset.columns[1 : -1].to_numpy()
print(gene.shape)
print(gene)
np.save('../data/allGene.npy', gene)

(17500,)
['A1BG' 'A1CF' 'A2ML1' ... 'ZYG11B' 'ZYX' 'ZZEF1']


In [8]:
data = myDataset.drop('submitter', axis = 1)

npData = data.to_numpy()
np.save('../data/allData.npy', npData)
print(f'shape = {npData.shape}, dtype = {type(npData)}')
print(npData)

shape = (185, 17501), dtype = <class 'numpy.ndarray'>
[[4 0 20409 ... 14846 3657 34.03]
 [7 0 121449 ... 20793 3996 32.62]
 [11 4 4051 ... 19369 2340 75.49]
 ...
 [9 3 324 ... 8071 6268 52.04]
 [4 1 27382 ... 14412 2061 7.95]
 [35 6 0 ... 16928 2493 8.31]]


# 二、Data preprocessing

## 1. Remove genes containing 0

In [9]:
import numpy as np
import pandas as pd

data = np.load('../data/allData.npy', allow_pickle = True)
data

array([[4, 0, 20409, ..., 14846, 3657, 34.03],
       [7, 0, 121449, ..., 20793, 3996, 32.62],
       [11, 4, 4051, ..., 19369, 2340, 75.49],
       ...,
       [9, 3, 324, ..., 8071, 6268, 52.04],
       [4, 1, 27382, ..., 14412, 2061, 7.95],
       [35, 6, 0, ..., 16928, 2493, 8.31]], dtype=object)

In [10]:
gene = np.load('../data/allGene.npy', allow_pickle=True)
print(gene.shape)
print(gene)
data[:, -1:] = np.where(data[:, -1:] > 12, 0, 1)
print(data.shape)
print(data)

(17500,)
['A1BG' 'A1CF' 'A2ML1' ... 'ZYG11B' 'ZYX' 'ZZEF1']
(185, 17501)
[[4 0 20409 ... 14846 3657 0]
 [7 0 121449 ... 20793 3996 0]
 [11 4 4051 ... 19369 2340 0]
 ...
 [9 3 324 ... 8071 6268 0]
 [4 1 27382 ... 14412 2061 1]
 [35 6 0 ... 16928 2493 1]]


In [11]:
feature = data[:, :-1].astype('float')
label = data[:, -1:].astype('int64')
print(feature.shape)
print(label.shape)

(185, 17500)
(185, 1)


In [12]:
# As long as the i-th gene expression of a patient is 0, the i-th gene will be discarded
reservedList = []

for i in range(feature.shape[1]):
    if feature[:, i].tolist().count(0) == 0: # retain
        reservedList.append(i)

feature = feature[:, reservedList]
gene = gene[reservedList]
print(feature.shape)
print(gene.shape)
print(gene)
print(feature)
np.save('../data/geneAfterDiscard_0.npy', gene)

np.save('../data/featureAfterDiscard_0.npy', np.append(feature, label, axis = 1))

(185, 11959)
(11959,)
['A2M' 'A4GALT' 'AAAS' ... 'ZYG11B' 'ZYX' 'ZZEF1']
[[17747.  7844.  2406. ...  1448. 14846.  3657.]
 [34953.  2996.  1182. ...  2241. 20793.  3996.]
 [ 4491.  9398.  2046. ...  1349. 19369.  2340.]
 ...
 [13188.  9383.  2233. ...  1603.  8071.  6268.]
 [ 7999.  2810.   844. ...  1057. 14412.  2061.]
 [34797.  2256.  1853. ...  1391. 16928.  2493.]]


## 2. Smote oversampling

In [13]:
# SMOTE
from imblearn.over_sampling import SMOTE
from collections import Counter

print(f'before overstamp :{Counter(label.reshape(-1))}')

# Set random state for recurrence

overstamp = SMOTE(random_state=2022)

x, y = overstamp.fit_resample(feature, label)

print(f'after overstamp :{Counter(y)}')

y = y.reshape((-1, 1))

oversampled_data = np.append(x, y, axis = 1)

np.save("../data/oversampled_data.npy", oversampled_data)

before overstamp :Counter({0: 148, 1: 37})
after overstamp :Counter({0: 148, 1: 148})
